1️⃣ Widgets


In [0]:
import ast

dbutils.widgets.text("table_metadata", "")
table_metadata_str = dbutils.widgets.get("table_metadata")

if not table_metadata_str:
    raise ValueError("table_metadata cannot be empty")

table_metadata = ast.literal_eval(table_metadata_str)

dbutils.widgets.text("table_parameters", "")
table_parameters_str = dbutils.widgets.get("table_parameters")

if not table_parameters_str:
    raise ValueError("table_parameters cannot be empty")

table_parameters = ast.literal_eval(table_parameters_str)

print(f"Table: {table_metadata.get('table_name')}")
print(f"Load type: {table_parameters.get('load_type')}")


Table: branches
Load type: FULL


2️⃣ Extract Variables

In [0]:
table_name = table_metadata.get("table_name")
source_path = table_metadata.get("source_path")
bronze_schema = table_metadata.get("bronze_schema")
source_system = table_metadata.get("source_system")
load_type = table_parameters.get("load_type")
primary_key = table_parameters.get("primary_key")
watermark_column = table_parameters.get("watermark_column")
table_id = int(table_metadata.get("table_id"))

bronze_table_fqn = f"banking.{bronze_schema}.{table_name}"

print(f"Table: {table_name}")
print(f"Source system: {source_system}")
print(f"Bronze table: {bronze_table_fqn}")

Table: branches
Source system: delta
Bronze table: banking.bronze.branches


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS banking.bronze

3️⃣ Get Last Watermark

In [0]:
last_watermark = None
watermark_column = table_parameters.get("watermark_column")

if load_type in ["APPEND", "MERGE"] and watermark_column:
    watermark_df = spark.sql(f"""
        SELECT last_watermark_value
        FROM banking.metadata.table_watermarks
        WHERE table_id = {table_metadata.get('table_id')}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["last_watermark_value"]

print(f"Last watermark: {last_watermark}")

Last watermark: None


In [0]:
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id")

if not run_id:
    import time
    run_id = str(int(time.time()))

print(f"Run ID: {run_id}")

Run ID: 123


In [0]:
start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
table_id = int(table_metadata.get("table_id"))

entry_exists = spark.sql(f"""
    SELECT 1 FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id} AND table_id = {table_id}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET layer = 'Bronze', start_time = TIMESTAMP('{start_time}'),
            end_time = NULL, status = 'INPROGRESS',
            number_of_records = NULL, error_message = NULL
        WHERE run_id = {run_id} AND table_id = {table_id}
    """)
else:
    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES ({run_id}, {table_id}, 'Bronze',
                TIMESTAMP('{start_time}'), NULL, 'INPROGRESS', NULL, NULL)
    """)

print(f"Pipeline run logged. run_id: {run_id}")

Pipeline run logged. run_id: 123


 Read source using Autoloader:

In [0]:
from pyspark.sql.functions import current_timestamp, col, lit

if source_system == "delta":
    source_df = (
        spark.read
        .table(source_path)
        .withColumn("insert_timestamp", current_timestamp())
        .withColumn("source_file_name", lit(source_path))
    )
else:
    source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"/Volumes/banking/source/volume/_schema/{table_name}")
    .option("header", "true")
    .load(source_path)
    .withColumn("insert_timestamp", current_timestamp())
    .withColumn("source_file_name", col("_metadata.file_path"))
)

print("Source read successfully.")

Source read successfully.


In [0]:
try:
    if source_system == "blob":
        (
            source_df.writeStream
            .format("delta")
            .option("checkpointLocation", f"/Volumes/banking/source/volume/_checkpoints/{table_name}")
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(bronze_table_fqn)
        )
        records_read = None

    elif source_system == "delta":
        (
            source_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(bronze_table_fqn)
        )
        records_read = source_df.count()

    else:
        raise ValueError("Unsupported source_system")

    print("Source → Bronze Load Completed Successfully.")

except Exception as e:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    error_message = str(e)

    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET end_time = TIMESTAMP('{end_time}'),
            status = 'FAILED',
            error_message = '{error_message.replace("'", "")}'
        WHERE table_id = {table_id} AND run_id = {run_id}
    """)
    raise

Source → Bronze Load Completed Successfully.


In [0]:
%sql
SELECT * FROM banking.bronze.customers

customer_id,first_name,last_name,date_of_birth,pan_number,email,phone_number,kyc_status,branch_code,created_at,updated_at,insert_timestamp,source_file_name
1001,Vikram,Gupta,1970-02-12,QAHFT6804A,vikram.gupta73@gmail.com,9708562115,VERIFIED,BR004,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1002,Anjali,Desai,1975-09-13,CGPOS7442A,anjali.desai13@gmail.com,9359134598,VERIFIED,BR001,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1003,Sneha,Reddy,1971-11-02,TEKHF7133L,sneha.reddy69@gmail.com,9720609124,VERIFIED,BR004,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1004,Suresh,Jain,1988-12-21,PUSNZ3586L,suresh.jain59@gmail.com,9757442398,VERIFIED,BR003,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1005,Pooja,Kumar,1995-03-18,PSBFH0212U,pooja.kumar16@gmail.com,9474221004,VERIFIED,BR001,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1006,Vikram,Mehta,1969-08-23,VEJRS6065H,vikram.mehta55@gmail.com,9117593027,VERIFIED,BR002,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1007,Kavita,Verma,2000-01-14,EJZQO6872B,kavita.verma17@gmail.com,9672564421,VERIFIED,BR004,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1008,Rahul,Mehta,1960-11-03,UKBXO7248U,rahul.mehta69@gmail.com,9459508941,PENDING,BR002,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1009,Vikram,Jain,1977-01-15,LGGOG5832E,vikram.jain74@gmail.com,9668710325,PENDING,BR001,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers
1010,Sneha,Kumar,1984-06-16,NCBCQ7403O,sneha.kumar58@gmail.com,9352903002,VERIFIED,BR003,null,null,2026-05-26T11:26:58.301Z,banking.banking.customers


In [0]:
%sql
SELECT * FROM banking.metadata.pipeline_runs

run_id,table_id,layer,start_time,end_time,status,number_of_records,error_message
1234,6,Bronze,2026-05-25T12:30:19.729Z,2026-05-25T12:31:05.944Z,SUCCESS,5000,null
1234,5,Bronze,2026-05-26T05:05:14.937Z,2026-05-26T05:20:42.015Z,SUCCESS,5000,null
123,5,Bronze,2026-05-20T05:11:20.191Z,2026-05-21T12:08:06.120Z,SUCCESS,4000,null
123,1,Bronze,2026-05-26T11:25:38.379Z,null,INPROGRESS,null,null
123,2,Bronze,2026-05-26T11:35:32.606Z,null,INPROGRESS,null,null
123,3,Bronze,2026-05-26T11:37:41.444Z,null,INPROGRESS,null,null
123,4,Bronze,2026-05-26T11:39:52.266Z,null,INPROGRESS,null,null


In [0]:
%sql
-- UPDATE banking.metadata.table_watermarks
-- SET last_watermark_value = '1900-01-01 00:00:00'
-- WHERE table_id = 5

num_affected_rows
1


In [0]:
%sql
SELECT COUNT(*) FROM banking.bronze.branches

COUNT(*)
4
